In [2]:
%run ../src/data/evaluate.py

1. Загрузка модели и теста.
Размер теста: 450 строк, из них аномалий: 28

2. Метрики (порог 0.5):
   PR-AUC: 0.943 (95% ДИ: 0.871 - 0.987)

3. Поиск оптимального порога (FN=10, FP=1):
   Оптимальный порог: 0.139
   При пороге 0.139: Recall=0.964, Precision=0.692


C:\Users\Charlie\mo_project\src\data\evaluate.py:103: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='importance', y='feature', data=imp_df, palette='viridis')



Графики сохранены в папку reports/ (3 шт).
Готово.


In [3]:
%run ../src/data/batch_eval.py

1. Загрузка модели.

2. Обработка новых порций.

batch_01.csv
Проверка качества: OK (Строк: 500)
Аномалий в порции: 29
PR-AUC: 0.924
Recall (порог 0.139): 1.000
PSI (Electricity_Consumed): 0.018

batch_02.csv
Проверка качества: OK (Строк: 500)
Аномалий в порции: 30
PR-AUC: 0.864
Recall (порог 0.139): 1.000
PSI (Electricity_Consumed): 0.027

batch_03.csv
Проверка качества: OK (Строк: 500)
Аномалий в порции: 31
PR-AUC: 0.927
Recall (порог 0.139): 0.968
PSI (Electricity_Consumed): 0.025

batch_04.csv
Проверка качества: OK (Строк: 500)
Аномалий в порции: 24
PR-AUC: 0.892
Recall (порог 0.139): 0.958
PSI (Electricity_Consumed): 0.016


In [4]:
%run ../src/data/test_quality.py

Проверка намеренно испорченного файла
Порция прошла проверку? False
Количество строк: 100
  [ОШИБКА] Q1 Timestamp уникален и без пропусков: найдено нарушений 11
  [ОШИБКА] Q2 Electricity_Consumed в диапазоне [0, 1]: найдено нарушений 6
  [ОШИБКА] Q5 Объем порции не менее 400 строк: найдено нарушений 1


In [5]:
import joblib
import numpy as np
import pandas as pd
from features import clean
from sklearn.metrics import precision_recall_curve, auc

model = joblib.load('../models/model_v1.joblib')
df = clean(pd.read_csv('../data/raw/initial_data.csv'))
df_test = df.iloc[int(len(df)*0.85):].copy()
y_test = (df_test['Anomaly_Label'] == 'Abnormal').astype(int)
X_test = df_test.drop(columns=['Anomaly_Label'])

def pr_auc_score(y_true, y_proba):
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    return auc(recall, precision)

y_proba_orig = model.predict_proba(X_test)[:, 1]
print(f"PR-AUC без шума: {pr_auc_score(y_test, y_proba_orig):.3f}")

X_noisy = X_test.copy()
noise = np.random.normal(1, 0.1, size=len(X_noisy))
X_noisy['Electricity_Consumed'] = X_noisy['Electricity_Consumed'] * noise
y_proba_noisy = model.predict_proba(X_noisy)[:, 1]
print(f"PR-AUC с шумом 10%: {pr_auc_score(y_test, y_proba_noisy):.3f}")

X_missing = X_test.copy()
X_missing.loc[X_missing.sample(frac=0.2).index, 'Electricity_Consumed'] = np.nan
y_proba_missing = model.predict_proba(X_missing)[:, 1]
print(f"PR-AUC с 20% пропусков: {pr_auc_score(y_test, y_proba_missing):.3f}")

PR-AUC без шума: 0.943
PR-AUC с шумом 10%: 0.878
PR-AUC с 20% пропусков: 0.865
